# Week 2 — The model is just a rule you can read

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Louai-Mohamed/flyrank-ml-internship/blob/main/notebooks/02_your_first_readable_model.ipynb?flush_cache=true)

You'll:
1. Write a **1-line hand rule** and rank pages with it.
2. Fit a **depth-2 decision tree** and `print` it — the model "learned" a readable if/else. Then compare — where does it beat your rule, and where doesn’t it?
3. See **why you never feed the outcome back in** — that's leakage.

The payoff isn't a high score. It's: *my intuition was rough, the model found the real signal, and I can read exactly what it found.*

## 0. Setup (Colab or local)

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Louai-Mohamed/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# The label: a page is 'declining' when its recent trend is down. Simple, honest starter label.
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
print(df.shape[0], "pages |  declining rate:", round(df["is_declining_label"].mean(), 3))

30000 pages |  declining rate: 0.542


## 1. A rule you write by hand: `stale x visible`
Intuition: a page worth reviewing is one that is **stale** (not updated in a while) **and** still **visible** (getting impressions). Rank those by how much exposure they have.

In [2]:
stale   = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)
df["hand_rule_score"] = stale * visible * df["impressions_90d"]

top10 = df.sort_values("hand_rule_score", ascending=False).head(10)
top10[["impressions_90d", "days_since_last_update", "avg_position", "ctr", "trend_direction"]]

,impressions_90d,days_since_last_update,avg_position,ctr,trend_direction
16751,61678,194,19.7,0.15,down
16514,59472,194,24.8,0.13,down
7021,25715,194,22.2,0.23,down
21268,13299,193,10.5,0.49,down
11489,7812,194,39.0,0.01,down
12045,7558,193,17.9,0.20,down
698,4590,194,31.0,0.00,down
5327,4556,194,16.4,0.33,down
26810,4429,194,25.3,0.38,down
20837,1697,193,15.8,0.12,down


We need a way to score any ranking. **Precision@K** = of the top K pages a ranking flags, what fraction are actually declining.

In [3]:
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

y = df["is_declining_label"].values
for k in (20, 50):
    print(f"Hand rule  Precision@{k}: {precision_at_k(df['hand_rule_score'], y, k):.3f}")

Hand rule  Precision@20: 0.900
Hand rule  Precision@50: 0.680


## 2. Let a model learn the rule — then read it
A **depth-2 decision tree** can only ask 3 yes/no questions. That constraint is the point: whatever it learns, you can read.

We give it a few **pre-decision** signals — never product flags.

In [4]:
from sklearn.tree import DecisionTreeClassifier, export_text

features = ["content_age_days", "days_since_last_update", "impressions_90d",
            "avg_position", "ctr", "word_count"]
X = df[features].replace([np.inf, -np.inf], np.nan).fillna(0)

tree = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42)
tree.fit(X, y)

print(export_text(tree, feature_names=features))

|--- impressions_90d <= 5.50
|   |--- avg_position <= 0.75
|   |   |--- class: 0
|   |--- avg_position >  0.75
|   |   |--- class: 0
|--- impressions_90d >  5.50
|   |--- content_age_days <= 312.50
|   |   |--- class: 1
|   |--- content_age_days >  312.50
|   |   |--- class: 0



That printout **is** the model — a human-readable if/else. Now rank pages by the tree's probability and score it the same way.

In [5]:
tree_score = tree.predict_proba(X)[:, 1]
for k in (20, 50):
    hr = precision_at_k(df["hand_rule_score"], y, k)
    tr = precision_at_k(tree_score, y, k)
    print(f"Precision@{k}:  hand rule {hr:.3f}   vs   tree {tr:.3f}")

Precision@20:  hand rule 0.900   vs   tree 0.800
Precision@50:  hand rule 0.680   vs   tree 0.740


Now read your own printout carefully — **the winner here depends on your run.** A depth-2 tree can only give four different scores (one per leaf), so the "top 50" is mostly one big block of tied pages, and different library versions break those ties differently. On some stacks the tree wins at Precision@50; on others the hand rule holds both. **Both results are real.** The stable lesson: a sharp human rule can be excellent at the very top of the list; a model's advantage — when it shows up — appears deeper, where simple rules run out of signal; and any comparison built on heavily tied scores is fragile. Saying exactly what YOUR run shows — instead of "the model is better" — is what honest evaluation sounds like.

## 3. Why you can't feed the outcome back in
Your label is `trend_direction == "down"`, and `trend_pct` is the exact percentage change that bucket is computed from — so it **is** the answer in disguise. Watch what happens if you feed it in as a feature:

In [6]:
X_leaky = df[features + ["trend_pct"]].replace([np.inf, -np.inf], np.nan).fillna(0)
leaky = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42).fit(X_leaky, y)
print(f"'Leaky' tree Precision@50: {precision_at_k(leaky.predict_proba(X_leaky)[:,1], y, 50):.3f}  <- looks amazing")
print(export_text(leaky, feature_names=features + ["trend_pct"]))

'Leaky' tree Precision@50: 1.000  <- looks amazing
|--- trend_pct <= -20.05
|   |--- word_count <= 212.00
|   |   |--- class: 1
|   |--- word_count >  212.00
|   |   |--- class: 1
|--- trend_pct >  -20.05
|   |--- trend_pct <= -19.95
|   |   |--- class: 0
|   |--- trend_pct >  -19.95
|   |   |--- class: 0



The tree just split on `trend_pct` and nailed the label — because the label is **derived from** `trend_pct`. That's **leakage**: the feature is the answer in disguise, and it teaches you nothing.

That's also why the starter data ships **only observable signals** — the product's own decision flags (health scores, "needs CTR fix", and so on) aren't included, so you can't accidentally train on them. You build from what was knowable *before* the outcome.

> Rule of thumb: if a feature would only be known *because someone already made the decision you're predicting*, it leaks. Leave it out.

## 4. 🔧 Your turn
- Change `max_depth` to 3 or 4 — does Precision@50 improve? Can you still read the tree?
- Swap in different features (drop `impressions_90d`, add `engagement_rate`). What does the tree choose to split on first?
- **Important caveat:** we scored *in-sample* here for teaching. The real pipeline uses **client-holdout** validation (`scripts/03_train_model.py`) so a client's pages never appear in both train and test. Re-run your comparison with a train/test split and see if the gap holds.

Write your experiment below.

### My experiment — how deep can it go, what does it split on, and does the gap survive unseen clients?

Three things, in order: (1) sweep `max_depth` and watch precision@K against readability, (2) drop
`impressions_90d` and add `engagement_rate` to see what the tree reaches for first, (3) re-run the
comparison on a **client-holdout split** — the part that decides whether any of this is real.

In [7]:
# 🔧 Experiment 1 — depth sweep (in-sample, same teaching setup as section 2)
from sklearn.metrics import roc_auc_score

rows = []
for depth in (2, 3, 4, 6):
    model = DecisionTreeClassifier(max_depth=depth, class_weight="balanced", random_state=42).fit(X, y)
    score = model.predict_proba(X)[:, 1]
    rows.append({
        "max_depth": depth,
        "leaves (readable rules)": model.get_n_leaves(),
        "precision@20": round(precision_at_k(score, y, 20), 3),
        "precision@50": round(precision_at_k(score, y, 50), 3),
        "roc_auc": round(roc_auc_score(y, score), 3),
    })

depth_sweep = pd.DataFrame(rows)
print("Scored IN-SAMPLE on all 30,000 pages — teaching comparison, not a performance claim:")
depth_sweep

Scored IN-SAMPLE on all 30,000 pages — teaching comparison, not a performance claim:


,max_depth,leaves (readable rules),precision@20,precision@50,roc_auc
0,2,4,0.80,0.74,0.651
1,3,8,0.70,0.72,0.677
2,4,16,0.60,0.68,0.693
3,6,57,0.95,0.96,0.728


**Read the sweep like this — it is two different stories in one table.** From depth 2 to 4, precision@K
**falls** (0.80 → 0.60 at @20, 0.74 → 0.68 at @50) while `roc_auc` **rises**: the extra splits sharpen the
whole ordering but blur the very top, where the reviewers actually work. Then depth 6 breaks the pattern
— 0.95 / 0.96, better than everything before it. That is not learning, it is **memorisation**: 57 leaves,
scored on the same 30,000 rows it was fitted on, and a printout nobody would read. In-sample scores reward
memorisation by construction, which is exactly why experiment 3 holds out whole clients.

In [8]:
# 🔧 Experiment 2 — drop impressions_90d, add engagement_rate. What does it split on first?
features_v2 = ["content_age_days", "days_since_last_update", "avg_position",
               "ctr", "word_count", "engagement_rate"]
X_v2 = df[features_v2].replace([np.inf, -np.inf], np.nan).fillna(0)

tree_v2 = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42)
tree_v2.fit(X_v2, y)
print(export_text(tree_v2, feature_names=features_v2, decimals=2))
print("precision@50 (in-sample):", round(precision_at_k(tree_v2.predict_proba(X_v2)[:, 1], y, 50), 3))
print("pages where avg_position == 0 (the dictionary's 'no data', not rank zero):",
      int((df["avg_position"] == 0).sum()))

|--- avg_position <= 0.55
|   |--- avg_position <= 0.15
|   |   |--- class: 0
|   |--- avg_position >  0.15
|   |   |--- class: 0
|--- avg_position >  0.55
|   |--- content_age_days <= 287.50
|   |   |--- class: 1
|   |--- content_age_days >  287.50
|   |   |--- class: 0

precision@50 (in-sample): 0.7
pages where avg_position == 0 (the dictionary's 'no data', not rank zero): 1205


**What it picked first — and it is a trap worth naming.** With exposure removed, the root split is
`avg_position <= 0.55`: that is not a ranking question, it is `avg_position == 0`, which
`docs/data-dictionary.md` defines as **no position data** (1,205 pages), not rank zero. A real zero and a
`fillna(0)` landed in the same bucket, so the tree's first and strongest question is "was this page
measured at all?". Only below that does it reach for `content_age_days` — the same variable my Week-1
discovery pointed at — while `engagement_rate`, the feature I added, never earns a question.

Two lessons: **a missingness flag is a signal whether you meant it or not**, and this is why the reference
pipeline builds explicit `has_*` flags instead of blind `fillna(0)`.

In [9]:
# 🔧 Experiment 3 — does the gap survive clients the model has never seen?
from sklearn.model_selection import GroupShuffleSplit


def precision_at_k_tie_aware(scores, labels, k, draws=100, seed=0):
    """Precision@k with ties broken at random, averaged over `draws` shuffles.
    Needed because a depth-2 tree only ever emits 4 distinct scores."""
    rng = np.random.default_rng(seed)
    scores = np.asarray(scores, dtype=float)
    return float(np.mean([
        labels[np.argsort(-(scores + rng.random(len(scores)) * 1e-9))[:k]].mean()
        for _ in range(draws)
    ]))


# A fairer hand rule: same idea as section 1, but rank-based so it ranks every page
# instead of flagging 17 of them. Weights are hand-picked, nothing is fitted.
df["rank_rule_score"] = (0.6 * df["impressions_90d"].rank(pct=True)
                         + 0.4 * df["days_since_last_update"].rank(pct=True))

records = []
for fold, (train_idx, test_idx) in enumerate(
        GroupShuffleSplit(n_splits=5, test_size=0.2, random_state=42)
        .split(df, groups=df["client_id"])):
    y_test = y[test_idx]
    rankings = {
        "1-line rule (section 1)": df["hand_rule_score"].values[test_idx],
        "ranked hand rule": df["rank_rule_score"].values[test_idx],
    }
    for depth in (2, 3, 4, 6):
        fitted = DecisionTreeClassifier(max_depth=depth, class_weight="balanced",
                                        random_state=42).fit(X.iloc[train_idx], y[train_idx])
        rankings[f"tree depth {depth}"] = fitted.predict_proba(X.iloc[test_idx])[:, 1]
    for name, score in rankings.items():
        records.append({"fold": fold, "ranking": name,
                        "precision@50": precision_at_k_tie_aware(score, y_test, 50),
                        "roc_auc": roc_auc_score(y_test, score),
                        "distinct_scores": len(np.unique(score))})

holdout = pd.DataFrame(records)
summary = (holdout.groupby("ranking")
                   .agg(mean_precision_at_50=("precision@50", "mean"),
                        sd_across_client_splits=("precision@50", "std"),
                        mean_roc_auc=("roc_auc", "mean"),
                        distinct_scores=("distinct_scores", "max"))
                   .round(3)
                   .sort_values("mean_roc_auc", ascending=False))
print("Client-holdout, 5 splits, 20% of clients held out each time:")
print(summary.to_string())
print(f"\nBase rate in the data (what a coin-flip ranking scores): {y.mean():.3f}")
print(f"Pages the 1-line rule actually flags: {int((df['hand_rule_score'] > 0).sum()):,} of {len(df):,}")

Client-holdout, 5 splits, 20% of clients held out each time:
                         mean_precision_at_50  sd_across_client_splits  mean_roc_auc  distinct_scores
ranking                                                                                              
tree depth 6                            0.602                    0.179         0.668               41
tree depth 3                            0.603                    0.114         0.667                8
tree depth 4                            0.610                    0.141         0.660               16
tree depth 2                            0.583                    0.108         0.647                4
ranked hand rule                        0.448                    0.153         0.572             3933
1-line rule (section 1)                 0.482                    0.096         0.500                4

Base rate in the data (what a coin-flip ranking scores): 0.542
Pages the 1-line rule actually flags: 17 of 30,000


#### Verdict — the direction holds, the precision number does not

**1. The section-1 rule is not a ranking at all.** It flags **17 pages out of 30,000**
(`days_since_last_update ≥ 180` *and* `≥ 500 impressions`). Its eye-catching in-sample Precision@50 comes
from 17 real hits followed by 33 pages tied at zero, ordered by row position — and held out it lands at
**ROC-AUC 0.500**, a coin flip by definition. Coverage check first, comparison second.

**2. A 'fairer' hand rule can do worse than nothing.** My rank-based rule (60% visibility rank + 40%
freshness rank) averaged a *lower* precision@50 than the base rate on unseen clients, at ROC-AUC ≈ 0.57.
Hand-built intuition ranked declining pages worse than random did. That is a result, not a failure.

**3. The trees carry a real but small signal.** Held out: precision@50 ≈ **0.58–0.61** for depths 2–4
against 0.45–0.48 for the two hand rules, and ROC-AUC ≈ **0.65–0.67** against 0.50–0.57. The lift survives
the split — but so does the noise: the spread across client splits (±0.11–0.14) is larger than the gap
between depth 2 and depth 4 (0.03), so 'depth 4 beats depth 2' is split noise, not a finding.

**4. And the 0.96 in-sample tree evaporates.** Depth 6, which looked like a miracle in experiment 1, scores
≈ 0.60 held out with the widest spread of all (±0.18) — memorisation does not transfer, and it is the least
readable model in the table.

**The honest sentence:** *on unseen clients, a readable depth-2/3 tree ranks declining pages better than a
hand rule (ROC-AUC ≈ 0.65 vs 0.50), the exact precision@50 moves by ±0.1 depending on which clients you
hold out, and going deeper buys nothing you can use.* That is why the reference pipeline adds feature
engineering and a forest before claiming its ~2.8x lift (0.24 → 0.68 Precision@50 on these library
versions) — and why, from Week 3 on, every comparison I run will use a client-holdout split and a
tie-aware metric.

### Save your work
**Colab:** *File → Save a copy in GitHub* (your submission) and *File → Save a copy in Drive*.

You now have the two core reflexes of applied ML: **discover before you model**, and **prefer a model you can read and can't fool**. That's the whole foundation the capstone builds on.